# 20_embed — SPECTER2 embeddings (task S1.6; reused at S4.0)

Thin driver: all logic lives in `citecurv.embed.specter2`. Runtime: **GPU** (a T4 is enough).

Reads `works.parquet` from the run folder on Drive, encodes title + [SEP] + abstract (title alone when the abstract is missing, FR3)
with `allenai/specter2_base` + the `allenai/specter2` proximity adapter, and writes `embeddings.npz`
(`ids` (N,) str, `vectors` (N, 768) float32). Encoding is chunked: finished chunks are saved to Drive and reused
if the session dies (plan 9.1 checkpoint-resume), so simply rerun all cells after a disconnect.

After the run, copy the printed record (throughput, GPU, versions) into `PROGRESS.md` (S1.6 verify line and the
unit-burn log).

Drive layout (docs/colab_runbook.md section 5): every artifact of one run lives in
`MyDrive/citecurv/<RUN>/` with `RUN` = `pilot` (S1.2) or `full` (S4.0), under the contract file names.
Set `RUN` in the parameters cell; every notebook (10 to 50) must use the same value.
Runtime version 26.07 (Python 3.12): the first code cell stops on any other Python.

In [ ]:
# ---- Python version gate (run this first) ----------------------------------------------------------
# Colab's default runtime has been Python 3.13 since 2026-08-25; the project pins Python 3.12. Fix: Runtime >
# Change runtime type > Runtime version 26.07 (Python 3.12), keep the T4 GPU, Save, then Run all. The choice does
# not persist between sessions. ALLOW_PY313 = True runs anyway (PROVISIONAL, untested for this stage; write it
# in PROGRESS.md if used).
import platform
import sys

ALLOW_PY313 = False

PY = sys.version_info[:2]
print("python  ", sys.version)
print("platform", platform.platform())
if PY != (3, 12) and not (PY == (3, 13) and ALLOW_PY313):
    raise RuntimeError(
        f"This notebook needs Python 3.12, found {platform.python_version()}. In Colab: Runtime > Change "
        "runtime type > Runtime version 26.07 (Python 3.12), Save, then Run all again.")

In [ ]:
# Embed-stage pins (configs/pins.md): adapters 1.3.0 requires transformers~=4.57.6, so pin both exactly
# instead of using the runtime's default transformers.
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
REPO_TAG = "<tag>"  # PROVISIONAL: set to the release tag that contains citecurv.embed.specter2 before running
%pip install -q "citecurv @ git+{REPO}@{REPO_TAG}" "adapters==1.3.0" "transformers==4.57.6"

import sys
import importlib.metadata as md

VERSIONS = {"python": sys.version.split()[0]}
for pkg in ("torch", "transformers", "adapters", "citecurv", "numpy", "pandas"):
    VERSIONS[pkg] = md.version(pkg)
for name, version in VERSIONS.items():
    print(f"{name:13s} {version}")
assert VERSIONS["adapters"] == "1.3.0" and VERSIONS["transformers"] == "4.57.6", (
    "embed pins not active: restart the runtime (Runtime > Restart session) and rerun this cell")

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive"
RUN = "pilot"                                  # "pilot" (S1.2) | "full" (S4.0): same value as in 10_data
RUN_DIR = Path(DRIVE_ROOT) / "citecurv" / RUN  # few large files only (plan 9.1)
WORKS = RUN_DIR / "works.parquet"
OUT = RUN_DIR / "embeddings.npz"
CHUNK_DIR = RUN_DIR / "embed_chunks"  # resume parts: ceil(N / CHUNK_SIZE) files, about 15 for 300K works
BATCH_SIZE = 32
CHUNK_SIZE = 20_000
assert WORKS.exists(), f"{WORKS} not found: run 10_data with RUN = {RUN!r} first"
print(WORKS)

In [ ]:
import torch

assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime > Change runtime type)"
DEVICE = "cuda"
GPU = torch.cuda.get_device_name(0)
print(GPU)

In [ ]:
import pandas as pd

works = pd.read_parquet(WORKS, columns=["work_id", "title", "abstract"])
has_abstract = works["abstract"].fillna("").str.strip().ne("")
print(f"{len(works):,} works; {has_abstract.mean():.1%} with abstract; "
      f"{(~has_abstract).sum():,} encoded from the title alone (FR3)")

In [ ]:
from citecurv.embed.specter2 import (
    encode_resumable,
    load_embeddings_npz,
    load_specter2,
    write_embeddings_npz,
)

tokenizer, model = load_specter2(DEVICE)  # base allenai/specter2_base + adapter allenai/specter2

In [ ]:
import time

t0 = time.perf_counter()
vectors, n_new = encode_resumable(
    works, CHUNK_DIR, chunk_size=CHUNK_SIZE,
    batch_size=BATCH_SIZE, model=model, tokenizer=tokenizer, device=DEVICE,
)
elapsed = time.perf_counter() - t0
docs_per_min = n_new / (elapsed / 60) if n_new else float("nan")
print(f"encoded {n_new:,} docs this session in {elapsed / 60:.1f} min -> {docs_per_min:,.0f} docs/min")
print(f"all chunks: {vectors.shape}; full slice of 300,000 docs at this rate: "
      f"{300_000 / docs_per_min / 60:.2f} GPU-hours")

In [ ]:
write_embeddings_npz(OUT, works["work_id"].to_numpy(), vectors)
ids, vecs = load_embeddings_npz(OUT)
assert len(ids) == len(works) and vecs.shape == (len(works), 768) and vecs.dtype == "float32"
print(OUT, vecs.shape, vecs.dtype)

Sanity check (plan S1.6): for three hand-picked triples (paper a, paper b from the same field, paper c from
another field) the cosine of (a, b) should exceed the cosine of (a, c). Fill `TRIPLES` with work ids.

In [ ]:
import numpy as np

TRIPLES = []  # e.g. [("W123", "W456", "W789"), ...] - three triples, ids from works.parquet
row = {w: i for i, w in enumerate(ids)}


def unit(w):
    v = vecs[row[w]].astype(np.float64)
    return v / np.linalg.norm(v)


for a, b, c in TRIPLES:
    same, cross = float(unit(a) @ unit(b)), float(unit(a) @ unit(c))
    print(a, b, c, f"same-field {same:.3f}  cross-field {cross:.3f}", "PASS" if same > cross else "FAIL")

In [ ]:
import datetime
import json

record = {
    "task": "S1.6",
    "date": datetime.date.today().isoformat(),
    "gpu": GPU,
    "n_works": len(works),
    "n_title_only": int((~has_abstract).sum()),
    "n_encoded_this_session": n_new,
    "minutes_this_session": round(elapsed / 60, 2),
    "docs_per_min": None if n_new == 0 else round(docs_per_min),
    "batch_size": BATCH_SIZE,
    "versions": VERSIONS,
}
print(json.dumps(record, indent=2))  # paste into PROGRESS.md; add the Colab unit burn by hand